# SHIELD — Master Notebook (Four Models + Ordinary CV + Editable SMOTE)

**PART A — Preprocessing:** clean data → label Bankrupt → split → Information Gain (top 20) → SMOTE.
RF/XGBoost use IG → SMOTE → classifier. Logistic Regression and RBF SVM use
IG → StandardScaler → SMOTE → classifier; the scaler is fitted on training data only.

**PART B — Development:** RandomForest, XGBoost, LogisticRegression and SVM_RBF across 11 protocols:
50:50 only, 70:30 only, 80:20 only, CV=5 only, CV=10 only, 50:50+CV=5, 70:30+CV=5,
80:20+CV=5, 50:50+CV=10, 70:30+CV=10 and 80:20+CV=10. **44 experiments per SMOTE ratio.**

**Restarted comparison:** `SMOTE_RATIO = 0.1`. After recording the 44 results, change this one variable
to **0.25**, then **0.5**, then **1.0**, rerunning from Setup each time: **176 results in total**.
SMOTE remains inside every applicable training fold. Validation/test rows are never resampled.

**Ordinary CV, no nested CV:** CV-only experiments evaluate predefined fixed parameters across
held-out folds. They do not tune on the full dataset and re-evaluate the selected settings on it.
Split+CV experiments use GridSearchCV only on their training partition, then evaluate on held-out data.
The `Best_Params` column is retained for compatibility; fixed-parameter runs report their configured
parameters, not a grid-search winner. `Parameter_Source` records this distinction.

**On hold:** classification-threshold tuning and broader RF tuning (`min_samples_leaf`, `max_features`).
All classifiers use their default `predict()` decision rule. SVM uses a decision score for ROC-AUC
and ROC curves, avoiding the extra CV used by probability calibration. Its score is not a probability.

**Output style preserved:** the one-line Accuracy/Precision/Recall/F1/ROC-AUC summary, classification
reports, grouped metric bars, confusion matrices, ROC curves and the original result-table columns.
Charts now compare four models; each confusion-matrix figure contains four panels.

**Run instructions:** upload this notebook to Colab, verify `RAW_PATH`, then use **Runtime → Run all**.
Only one ratio runs at a time. Reports and checkpoints use a new versioned directory, separated by
ratio, data, configuration and package versions. Previous nested-CV results are not overwritten.
The earlier notebook is archived locally under `notebooks/archive/SHIELD_Master_Validated_SMOTE_nested_v1.ipynb`.
CV-only folds and complete experiments are checkpointed for resume with the same configuration.


---
# PART A — PREPROCESSING
Part A demonstrates one 80:20 split. Part B fits fresh pipelines directly on the original cleaned features;
it does **not** use the demo's feature-selected or SMOTE-resampled CSVs as CV inputs.


## A1. Setup and the editable SMOTE variable


In [ ]:
!pip install -q imbalanced-learn xgboost openpyxl

import os
import random
import json
import hashlib
import platform
import time
import time
import time
import tempfile
import importlib.metadata
from functools import partial
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, ParameterGrid, ParameterGrid, ParameterGrid
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            roc_auc_score, roc_curve, confusion_matrix, classification_report)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

# ==========================================================
# CHANGE THIS ONE VARIABLE TO RUN A DIFFERENT SMOTE STRENGTH.
# 0.1 = minority count / majority count after resampling = 1:10.
# Later comparisons: 0.25, 0.5, 1.0. This is NOT a probability threshold.
SMOTE_RATIO = 0.1
# ==========================================================

SEED = 42
K_FEATURES = 20
SMOTE_K_NEIGHBORS = 5
N_JOBS = -1  # GridSearchCV parallelism; each individual model uses one worker.
USE_PIPELINE_CACHE = True  # Reuse IG/SMOTE fits for identical training folds across candidates.
NOTEBOOK_VERSION = "shield-four-model-ordinary-cv-v2"
TARGET_COL = "Bankrupt"
LABEL_MAP = {0: "Not Bankrupt", 1: "Bankrupt"}

assert 0 < SMOTE_RATIO <= 1, "SMOTE_RATIO must be greater than 0 and at most 1."
random.seed(SEED)
np.random.seed(SEED)

drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/SHIELD_Project/data'
BASE_REPORT_DIR = '/content/drive/MyDrive/SHIELD_Project/reports'
BASE_MODEL_DIR = '/content/drive/MyDrive/SHIELD_Project/models'
RAW_PATH = f'{drive_path}/bankruptcy_italian_companies_2023.xlsx'
PACKAGE_VERSIONS = {
    name: importlib.metadata.version(name)
    for name in ['numpy', 'pandas', 'scikit-learn', 'imbalanced-learn', 'xgboost', 'joblib']
}
print('Setup complete. SEED =', SEED)
print('SMOTE_RATIO =', SMOTE_RATIO, '(minority / majority after resampling)')
print('Package versions:', PACKAGE_VERSIONS)


## A2. Load Raw Data
*Verification: shape and first rows of the untouched raw dataset.*


In [ ]:
df_raw = pd.read_excel(RAW_PATH)
print('=== RAW DATA LOADED ===')
print('Shape:', df_raw.shape)
display(df_raw.head())


## A3. Clean Data
Drop `Company` and constant predictor columns, matching the original cleaning flow. Validate the target,
numeric predictors, missing values, and infinities. Report duplicates without silently deleting companies
that happen to share financial ratios. No additional outlier removal or scaling is introduced in this version.


In [ ]:
df = df_raw.copy()
before_shape = df.shape
assert TARGET_COL in df.columns, f'Missing target column: {TARGET_COL}'
dropped_cols = []
if 'Company' in df.columns:
    df = df.drop(columns=['Company'])
    dropped_cols.append('Company')
constant_cols = [c for c in df.columns if c != TARGET_COL and df[c].nunique(dropna=False) <= 1]
if constant_cols:
    df = df.drop(columns=constant_cols)
    dropped_cols.extend(constant_cols)

assert df.isnull().sum().sum() == 0, 'Missing values found. Investigate before continuing.'
assert set(df[TARGET_COL].unique()) == {0, 1}, 'Target must contain exactly 0 and 1.'
predictor_cols = df.columns.drop(TARGET_COL)
non_numeric = df[predictor_cols].select_dtypes(exclude=[np.number]).columns.tolist()
assert not non_numeric, f'Non-numeric predictors found: {non_numeric}'
assert np.isfinite(df[predictor_cols].to_numpy(dtype=float)).all(), 'Infinite predictor values found.'
assert len(predictor_cols) >= K_FEATURES, 'Fewer predictors than K_FEATURES.'

print('=== DATA CLEANING VERIFICATION ===')
print(f'Shape before cleaning: {before_shape}')
print(f'Shape after cleaning:  {df.shape}')
print(f'Columns dropped ({len(dropped_cols)}): {dropped_cols}')
print('No missing values confirmed.')
duplicate_count = int(df.duplicated().sum())
print(f'Duplicate cleaned rows: {duplicate_count} (reported; not automatically removed)')
if 'Company' in df_raw:
    repeated_ids = int(df_raw['Company'].duplicated().sum())
    print(f'Repeated Company identifiers: {repeated_ids}')
    if repeated_ids:
        raise ValueError('Repeated Company identifiers require investigation/grouped splitting before continuing.')


## A4. Target Column and Labelling
`Bankrupt`: 0 = Not Bankrupt; 1 = Bankrupt.


In [ ]:
y_full = df[TARGET_COL].astype(int)
X_full = df.drop(columns=[TARGET_COL])
label_counts = y_full.value_counts().sort_index()
assert label_counts[1] < label_counts[0], 'This notebook expects Bankrupt (1) to be the minority class.'
assert SMOTE_RATIO > label_counts[1] / label_counts[0], 'SMOTE_RATIO is below the existing minority/majority ratio.'

print('=== TARGET LABELLING VERIFICATION ===')
for cls, count in label_counts.items():
    print(f'  {cls} ({LABEL_MAP[cls]}): {count} rows ({count / len(y_full) * 100:.2f}%)')
print(f'Total rows: {len(y_full)}')
print(f'Features (X) shape: {X_full.shape}')
print(f'Imbalance ratio (Bankrupt:Not Bankrupt) = 1:{label_counts[0] / label_counts[1]:.1f}')

# Hash the actual cleaned data, including feature names, order, dtypes, and values.
data_hasher = hashlib.sha256()
data_hasher.update(pd.util.hash_pandas_object(df, index=True).values.tobytes())
data_hasher.update(json.dumps([(str(c), str(df[c].dtype)) for c in df.columns]).encode())
DATA_FINGERPRINT = data_hasher.hexdigest()
print('Cleaned data fingerprint:', DATA_FINGERPRINT[:16])


## A5. Train-Test Split (80:20 demonstration)
*Verification: shapes and stratified class distributions.*


In [ ]:
X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X_full, y_full, test_size=0.20, random_state=SEED, stratify=y_full)
print('=== SPLIT VERIFICATION (80:20 demo) ===')
print(f'X_train: {X_train_demo.shape}, X_test: {X_test_demo.shape}')
print('y_train distribution:'); print(y_train_demo.value_counts().sort_index())
print('y_test distribution:'); print(y_test_demo.value_counts().sort_index())


## A6. Information Gain — Top 20 Features (demonstration)
Fit only on the demo training set, then transform the test set. The development pipeline repeats this
selection within each CV training fold. The final selected 20 may differ between folds and experiments;
the deployed final model has one saved, fixed feature list.


In [ ]:
def make_selector():
    return SelectKBest(score_func=partial(mutual_info_classif, random_state=SEED), k=K_FEATURES).set_output(transform='pandas')

selector_demo = make_selector()
X_train_sel_demo = selector_demo.fit_transform(X_train_demo, y_train_demo)
X_test_sel_demo = selector_demo.transform(X_test_demo)
selected_cols_demo = X_full.columns[selector_demo.get_support()].tolist()
ig_full_df = pd.DataFrame({'Feature': X_full.columns, 'IG_Score': selector_demo.scores_})
ig_full_df = ig_full_df.sort_values(by='IG_Score', ascending=False).reset_index(drop=True)

print('=== INFORMATION GAIN VERIFICATION (80:20 demo) ===')
print(f'Selected {len(selected_cols_demo)} features (target: {K_FEATURES})')
print('\nTop 20 Selected Features (ranked by IG score):')
display(ig_full_df.head(K_FEATURES))
print(f'\nX_train_selected shape: {X_train_sel_demo.shape}')
print(f'X_test_selected shape:  {X_test_sel_demo.shape}')


## A7. SMOTE — Training Set Only (demonstration)
Use the global ratio, not automatic 50:50 balancing. At `0.1`, the resampled minority count is about
one tenth of the majority count. The test set remains untouched.


In [ ]:
print('=== SMOTE VERIFICATION (80:20 demo) ===')
print('SMOTE_RATIO:', SMOTE_RATIO)
print('Train set BEFORE SMOTE:')
print(y_train_demo.value_counts().sort_index())
X_train_bal_demo, y_train_bal_demo = SMOTE(
    sampling_strategy=SMOTE_RATIO, random_state=SEED, k_neighbors=SMOTE_K_NEIGHBORS
).fit_resample(X_train_sel_demo, y_train_demo)
print('\nTrain set AFTER SMOTE:')
print(pd.Series(y_train_bal_demo).value_counts().sort_index())
print(f'\nTrain shape before: {X_train_sel_demo.shape} -> after: {X_train_bal_demo.shape}')
print('\nTest set (UNCHANGED, still imbalanced):')
print(y_test_demo.value_counts().sort_index())
assert len(X_test_sel_demo) == len(X_test_demo)


## A8. Preprocessing Summary


In [ ]:
print('=== PREPROCESSING PIPELINE SUMMARY ===')
print(f'1. Raw data loaded:        {df_raw.shape}')
print(f'2. After cleaning:         {df.shape} (dropped: {dropped_cols})')
print(f'3. Target labelled:        {TARGET_COL} -> 0/1 ({LABEL_MAP})')
print(f'4. Split (80:20 demo):     train {X_train_demo.shape}, test {X_test_demo.shape}')
print(f'5. IG top-{K_FEATURES}:            selected {len(selected_cols_demo)} features')
print(f'6. SMOTE (train only):     {X_train_sel_demo.shape} -> {X_train_bal_demo.shape}')
print('\nDevelopment uses X_full/y_full, not these preprocessed demo arrays.')


---
# PART B — DEVELOPMENT

**Split only:** fit one training pipeline using predefined fixed model parameters; evaluate on the
held-out 50%, 30% or 20% partition. These runs do not perform parameter selection.

**Split + CV:** reserve the test partition. GridSearchCV fits IG → optional scaling → SMOTE → model
inside each training fold, selects parameters using Bankrupt-class F1, then refits on the entire
training partition. Headline metrics use the held-out test partition. `CV_F1_Mean (%)` is the winning
mean training-validation score, not an independent test score.

**CV only (ordinary, non-nested):** fit the same predefined parameters separately in every fold.
Pool held-out predictions for headline metrics and report mean fold F1 in `CV_F1_Mean (%)`.
There is no inner grid search, no full-dataset parameter selection and no nested CV.
A final full-data fit using the same fixed parameters is saved for export only; its training
predictions are never used for evaluation. Fold-level metrics/features are saved separately.

`Best_Params` retains the historical column name: fixed runs list their predefined settings;
split+CV runs list the grid winner. Check `Parameter_Source` and `Evaluation` in the full CSV.
All four models receive matching raw rows and folds within each protocol. The fixed parameters
and grids must be decided before evaluation; changing them in response to CV-only scores makes
that evaluation exploratory. Scores on different held-out groups are not interchangeable.

SVM_RBF ROC-AUC uses its continuous decision margin; other models use their Bankrupt-class
probability score. Scores pooled across CV folds are retained for the original headline style;
mean fold ROC-AUC is also saved because score scales can vary between fitted models.
Previously inspected test results and repeated model/SMOTE comparisons are exploratory.


## B1. Four-Model Configuration, 11 Protocols and New Checkpoint Isolation


In [ ]:
PROTOCOLS = [
    {'name': '50-50 only',     'slug': '5050_only', 'split': 0.50, 'cv': None},
    {'name': '70-30 only',     'slug': '7030_only', 'split': 0.30, 'cv': None},
    {'name': '80-20 only',     'slug': '8020_only', 'split': 0.20, 'cv': None},
    {'name': 'CV=5 only',      'slug': 'cv5_only',  'split': None, 'cv': 5},
    {'name': 'CV=10 only',     'slug': 'cv10_only', 'split': None, 'cv': 10},
    {'name': '50-50 + CV=5',   'slug': '5050_cv5',  'split': 0.50, 'cv': 5},
    {'name': '70-30 + CV=5',   'slug': '7030_cv5',  'split': 0.30, 'cv': 5},
    {'name': '80-20 + CV=5',   'slug': '8020_cv5',  'split': 0.20, 'cv': 5},
    {'name': '50-50 + CV=10',  'slug': '5050_cv10', 'split': 0.50, 'cv': 10},
    {'name': '70-30 + CV=10',  'slug': '7030_cv10', 'split': 0.30, 'cv': 10},
    {'name': '80-20 + CV=10',  'slug': '8020_cv10', 'split': 0.20, 'cv': 10},
]
MODELS = ['RandomForest', 'XGBoost', 'LogisticRegression', 'SVM_RBF']
SCALED_MODELS = ['LogisticRegression', 'SVM_RBF']
MODEL_LABELS = {'RandomForest': 'Random Forest', 'XGBoost': 'XGBoost',
                'LogisticRegression': 'Logistic Regression', 'SVM_RBF': 'SVM (RBF)'}
SCORE_TYPES = {name: ('decision_function' if name == 'SVM_RBF' else 'predict_proba')
               for name in MODELS}

# Fixed baselines are specified before evaluation, not copied from historical winners.
# These settings are used for BOTH split-only and ordinary CV-only experiments.
FIXED_PARAMS = {
    'RandomForest': {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 2},
    'XGBoost': {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    'LogisticRegression': {'C': 1.0},
    'SVM_RBF': {'C': 1.0, 'gamma': 'scale'},
}
ESTIMATOR_OPTIONS = {
    'RandomForest': {'random_state': SEED, 'n_jobs': 1},
    'XGBoost': {'random_state': SEED, 'n_jobs': 1, 'eval_metric': 'logloss'},
    'LogisticRegression': {'solver': 'lbfgs', 'max_iter': 5000, 'random_state': SEED},
    # Default probability estimation is disabled; ROC-AUC uses decision_function.
    'SVM_RBF': {'kernel': 'rbf', 'cache_size': 512, 'random_state': SEED},
}
# RF/XGBoost grids are unchanged. New models have compact grids.
# Only split+CV experiments use these grids. Broader RF tuning remains ON HOLD.
PARAM_GRIDS = {
    'RandomForest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [6, 10, None],
        'model__min_samples_split': [2, 5],
    },
    'XGBoost': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [3, 6, 9],
        'model__learning_rate': [0.01, 0.1, 0.2],
    },
    'LogisticRegression': {'model__C': [0.01, 0.1, 1.0, 10.0]},
    'SVM_RBF': {'model__C': [0.1, 1.0, 10.0],
                'model__gamma': ['scale', 0.01, 0.1]},
}

RATIO_TAG = format(float(SMOTE_RATIO), '.12g').replace('.', 'p')
RUN_SPEC = {
    'notebook_version': NOTEBOOK_VERSION, 'data_fingerprint': DATA_FINGERPRINT,
    'smote_ratio': float(SMOTE_RATIO), 'seed': SEED, 'k_features': K_FEATURES,
    'smote_k_neighbors': SMOTE_K_NEIGHBORS, 'protocols': PROTOCOLS, 'models': MODELS,
    'fixed_params': FIXED_PARAMS, 'param_grids': PARAM_GRIDS,
    'estimator_options': ESTIMATOR_OPTIONS, 'scaled_models': SCALED_MODELS,
    'pipeline_policy': 'IG -> optional StandardScaler -> SMOTE -> classifier; train-fold fitting',
    'score_types': SCORE_TYPES,
    'cv_only_policy': 'ordinary stratified CV with predefined fixed parameters; no grid search',
    'split_cv_policy': 'GridSearchCV on training partition only; scoring=f1',
    'splitter': f'StratifiedKFold(shuffle=True, random_state={SEED})',
    'threshold_policy': 'default predict(); threshold tuning on hold',
    'packages': PACKAGE_VERSIONS, 'python': platform.python_version(),
}
RUN_ID = hashlib.sha256(json.dumps(RUN_SPEC, sort_keys=True).encode()).hexdigest()[:16]
report_dir = f'{BASE_REPORT_DIR}/ordinary_cv_four_models_smote_{RATIO_TAG}/{RUN_ID}'
ckpt_dir = f'{report_dir}/checkpoints_master'
model_dir = f'{BASE_MODEL_DIR}/ordinary_cv_four_models_smote_{RATIO_TAG}/{RUN_ID}'
for directory in [report_dir, ckpt_dir, model_dir]:
    os.makedirs(directory, exist_ok=True)
with open(f'{report_dir}/run_config.json', 'w') as f:
    json.dump(RUN_SPEC, f, indent=2)
pipeline_memory = joblib.Memory(
    location=f'{tempfile.gettempdir()}/shield_pipeline_cache/{RUN_ID}', verbose=0
) if USE_PIPELINE_CACHE else None

# This demonstration belongs only to the tree-model preprocessing path.
# LR/SVM are scaled within their own training pipelines, not using demo arrays.
demo_dir = f'{report_dir}/demo_preprocessed_8020'
os.makedirs(demo_dir, exist_ok=True)
X_train_bal_demo.to_csv(f'{demo_dir}/X_train_processed_8020.csv', index=False)
pd.Series(y_train_bal_demo, name=TARGET_COL).to_csv(f'{demo_dir}/y_train_processed_8020.csv', index=False)
X_test_sel_demo.to_csv(f'{demo_dir}/X_test_processed_8020.csv', index=False)
y_test_demo.to_csv(f'{demo_dir}/y_test_processed_8020.csv', index=False)
pd.DataFrame({'Feature': selected_cols_demo}).to_csv(f'{demo_dir}/selected_features.csv', index=False)
df.to_excel(f'{demo_dir}/preprocessed_dataset.xlsx', index=False)

print(f'{len(PROTOCOLS)} protocols x {len(MODELS)} models = {len(PROTOCOLS) * len(MODELS)} total experiments')
print('SMOTE_RATIO:', SMOTE_RATIO)
print('CV-only: ordinary CV with predefined fixed parameters (no nested search).')
for model_name in MODELS:
    print(f'  {model_name}: fixed={FIXED_PARAMS[model_name]}, '
          f'split+CV grid={len(list(ParameterGrid(PARAM_GRIDS[model_name])))} candidates')
print('Run ID:', RUN_ID)
print('Reports/checkpoints:', report_dir)
print('After recording this ratio, rerun Setup onward with 0.25, then 0.5, then 1.0.')


## B2. Experiment Runner (fold-local IG/scaling/SMOTE and Ordinary CV-only)


In [ ]:
def make_estimator(model_name, params=None):
    constructors = {'RandomForest': RandomForestClassifier, 'XGBoost': XGBClassifier,
                    'LogisticRegression': LogisticRegression, 'SVM_RBF': SVC}
    if model_name not in constructors:
        raise ValueError(model_name)
    return constructors[model_name](**{**ESTIMATOR_OPTIONS[model_name], **(params or {})})

def make_training_pipeline(model_name, params=None):
    steps = [('ig', make_selector())]
    if model_name in SCALED_MODELS:
        steps.append(('scale', StandardScaler().set_output(transform='pandas')))
    steps.extend([
        ('smote', SMOTE(sampling_strategy=SMOTE_RATIO, random_state=SEED,
                        k_neighbors=SMOTE_K_NEIGHBORS)),
        ('model', make_estimator(model_name, params)),
    ])
    return ImbPipeline(steps, memory=pipeline_memory)

def make_cv(folds):
    return StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)

def make_grid(model_name, folds):
    return GridSearchCV(make_training_pipeline(model_name), PARAM_GRIDS[model_name],
                        cv=make_cv(folds), scoring='f1', n_jobs=N_JOBS,
                        refit=True, error_score='raise')

def check_resampling_feasible(y, context):
    counts = pd.Series(y).value_counts()
    if set(counts.index) != {0, 1}:
        raise ValueError(f'{context}: both target classes must be present.')
    desired_minority = int(SMOTE_RATIO * int(counts[0]))
    if desired_minority <= counts[1]:
        raise ValueError(f'{context}: SMOTE_RATIO={SMOTE_RATIO} would not add minority samples; '
                         'increase the ratio for this dataset/fold.')
    if counts[1] <= SMOTE_K_NEIGHBORS:
        raise ValueError(f'{context}: too few bankrupt samples for SMOTE k_neighbors={SMOTE_K_NEIGHBORS}.')

def check_cv_feasible(X, y, folds, context):
    counts = pd.Series(y).value_counts()
    if counts.min() < folds:
        raise ValueError(f'{context}: fewer minority samples than CV folds.')
    for fold_id, (train_idx, _) in enumerate(make_cv(folds).split(X, y), start=1):
        check_resampling_feasible(y.iloc[train_idx], f'{context}, training fold {fold_id}')

def compute_metrics(y_true, y_pred, y_score):
    return {
        'Accuracy (%)': accuracy_score(y_true, y_pred) * 100,
        'Precision (%)': precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'Recall (%)': recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'F1-Score (%)': f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'ROC-AUC (%)': roc_auc_score(y_true, y_score) * 100,
    }

def predict_outputs(fitted_pipeline, X):
    classifier = fitted_pipeline.named_steps['model']
    classes = classifier.classes_.tolist()
    assert classes == [0, 1], f'Unexpected target order: {classes}'
    y_pred = fitted_pipeline.predict(X)
    if isinstance(classifier, SVC):
        y_score = fitted_pipeline.decision_function(X)  # Larger margin means Bankrupt (1).
    else:
        y_score = fitted_pipeline.predict_proba(X)[:, classes.index(1)]
    return y_pred, np.asarray(y_score, dtype=float)

def selected_features(fitted_pipeline):
    selector = fitted_pipeline.named_steps['ig']
    return selector.feature_names_in_[selector.get_support()].tolist()

def checkpoint_paths(slug, model_name):
    base = f'{ckpt_dir}/{slug}_{model_name}'
    return {'result': f'{base}_result.json', 'preds': f'{base}_preds.npz',
            'model': f'{base}_model.joblib'}

def load_checkpoint(slug, model_name):
    paths = checkpoint_paths(slug, model_name)
    if not all(os.path.exists(p) for p in paths.values()):
        return None
    with open(paths['result']) as f:
        result = json.load(f)
    if result.get('Run_ID') != RUN_ID:
        return None
    with np.load(paths['preds'], allow_pickle=False) as arrays:
        return result, arrays['y_true'].copy(), arrays['y_pred'].copy(), arrays['y_score'].copy()

def save_checkpoint(slug, model_name, result, y_true, y_pred, y_score, fitted_model):
    paths = checkpoint_paths(slug, model_name)
    # Save the result last: it marks an experiment as complete.
    joblib.dump(fitted_model, paths['model'])
    np.savez_compressed(paths['preds'], y_true=np.asarray(y_true),
                        y_pred=np.asarray(y_pred), y_score=np.asarray(y_score))
    with open(paths['result'], 'w') as f:
        json.dump(result, f, indent=2)

def run_ordinary_cv(model_name, config):
    folds = config['cv']
    check_cv_feasible(X_full, y_full, folds, 'Ordinary fixed-parameter CV')
    y_pred = np.empty(len(y_full), dtype=int)
    y_score = np.empty(len(y_full), dtype=float)
    visited = np.zeros(len(y_full), dtype=bool)
    fold_records = []
    for fold_id, (train_idx, val_idx) in enumerate(make_cv(folds).split(X_full, y_full), start=1):
        fold_base = f"{ckpt_dir}/{config['slug']}_{model_name}_fold_{fold_id}"
        fold_json, fold_npz = f'{fold_base}.json', f'{fold_base}.npz'
        cached = None
        if os.path.exists(fold_json) and os.path.exists(fold_npz):
            with open(fold_json) as f:
                record = json.load(f)
            if record.get('Run_ID') == RUN_ID:
                with np.load(fold_npz, allow_pickle=False) as arrays:
                    if np.array_equal(arrays['val_idx'], val_idx):
                        cached = arrays['y_pred'].copy(), arrays['y_score'].copy(), record
        if cached is not None:
            fold_pred, fold_score, record = cached
            print(f'  [FOLD CHECKPOINT] {model_name} | {config["name"]} | fold {fold_id}/{folds}')
        else:
            fold_start = time.perf_counter()
            print(f'  [ORDINARY CV] {model_name} | {config["name"]} | fold {fold_id}/{folds}')
            fitted_fold = make_training_pipeline(model_name, FIXED_PARAMS[model_name])
            fitted_fold.fit(X_full.iloc[train_idx], y_full.iloc[train_idx])
            fold_pred, fold_score = predict_outputs(fitted_fold, X_full.iloc[val_idx])
            record = {'Run_ID': RUN_ID, 'Fold': fold_id,
                      'Configured_Params': FIXED_PARAMS[model_name],
                      'Parameter_Source': 'Predefined fixed parameters; no tuning',
                      'Score_Type': SCORE_TYPES[model_name],
                      'Selected_Features': selected_features(fitted_fold),
                      'Train_Count': len(train_idx), 'Validation_Count': len(val_idx),
                      'Elapsed_Seconds': time.perf_counter() - fold_start,
                      **compute_metrics(y_full.iloc[val_idx], fold_pred, fold_score)}
            np.savez_compressed(fold_npz, val_idx=val_idx, y_pred=fold_pred, y_score=fold_score)
            with open(fold_json, 'w') as f:
                json.dump(record, f, indent=2)
        assert not visited[val_idx].any(), 'A held-out CV row was predicted twice.'
        y_pred[val_idx], y_score[val_idx], visited[val_idx] = fold_pred, fold_score, True
        fold_records.append(record)
    assert visited.all(), 'Every row must receive exactly one held-out CV prediction.'
    pd.DataFrame(fold_records).to_csv(
        f"{report_dir}/{config['slug']}_{model_name}_fold_results.csv", index=False)

    # One full-data fit with the SAME fixed settings, for export only (no grid search).
    print(f'  [FIXED FINAL REFIT] {model_name} | {config["name"]} | all cleaned data')
    check_resampling_feasible(y_full, 'Final fixed full-data refit')
    fitted_model = make_training_pipeline(model_name, FIXED_PARAMS[model_name])
    fitted_model.fit(X_full, y_full)
    return y_pred, y_score, fold_records, fitted_model

def run_experiment(model_name, config):
    started = time.perf_counter()
    split_size, cv_folds = config['split'], config['cv']
    result = {'Model': model_name, 'Experiment': config['name'], 'Split': split_size, 'CV': cv_folds,
              'SMOTE_Ratio': float(SMOTE_RATIO), 'Run_ID': RUN_ID,
              'Score_Type': SCORE_TYPES[model_name],
              'Scaling': 'StandardScaler before SMOTE' if model_name in SCALED_MODELS else 'None'}

    if split_size is None:
        y_pred, y_score, fold_records, fitted_model = run_ordinary_cv(model_name, config)
        y_true = y_full.to_numpy()
        fold_scores = [r['F1-Score (%)'] for r in fold_records]
        result.update(compute_metrics(y_true, y_pred, y_score))
        result.update({'Best_Params': FIXED_PARAMS[model_name],
                       'CV_F1_Mean (%)': float(np.mean(fold_scores)),
                       'CV_F1_Std (%)': float(np.std(fold_scores, ddof=1)),
                       'CV_ROC_AUC_Mean (%)': float(np.mean([r['ROC-AUC (%)'] for r in fold_records])),
                       'Parameter_Source': 'Predefined fixed parameters; no tuning',
                       'Evaluation': 'Ordinary CV held-out predictions (fixed parameters)'})
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X_full, y_full, test_size=split_size, random_state=SEED, stratify=y_full)
        if cv_folds is None:
            check_resampling_feasible(y_train, 'Holdout training set')
            fitted_model = make_training_pipeline(model_name, FIXED_PARAMS[model_name])
            fitted_model.fit(X_train, y_train)
            result['Best_Params'] = FIXED_PARAMS[model_name]
            result['CV_F1_Mean (%)'] = None
            result['Parameter_Source'] = 'Predefined fixed parameters; no tuning'
        else:
            check_cv_feasible(X_train, y_train, cv_folds, 'Holdout training CV')
            grid = make_grid(model_name, cv_folds)
            grid.fit(X_train, y_train)
            fitted_model = grid.best_estimator_
            result['Best_Params'] = grid.best_params_
            result['CV_F1_Mean (%)'] = grid.best_score_ * 100
            result['Parameter_Source'] = 'GridSearchCV on training partition only'
            pd.DataFrame(grid.cv_results_).to_csv(
                f"{report_dir}/{config['slug']}_{model_name}_grid_results.csv", index=False)
        y_pred, y_score = predict_outputs(fitted_model, X_test)
        y_true = y_test.to_numpy()
        result.update(compute_metrics(y_true, y_pred, y_score))
        result['Evaluation'] = 'Held-out test partition; excluded from this fit/search'

    result['Selected_Features'] = selected_features(fitted_model)
    result['Evaluation_Count'] = len(y_true)
    result['Bankrupt_Support'] = int(np.sum(y_true == 1))
    result['Elapsed_Seconds'] = time.perf_counter() - started
    return result, y_true, y_pred, y_score, fitted_model

print('Runner defined: IG → optional scaling → SMOTE → model inside every CV training fold.')
print('CV-only is ordinary CV with fixed parameters. Split+CV tunes on training data only.')
print('SVM ROC-AUC uses decision scores; threshold tuning is ON HOLD.')


## B3. Run All 44 Experiments
Run in the requested protocol order, with RF, XGBoost, Logistic Regression and RBF SVM in each group.
The new version/data/configuration fingerprint prevents old nested-CV checkpoints from being reused.
Completed experiments and CV-only folds can resume from matching checkpoints.


In [ ]:
all_results = []
predictions = {cfg['slug']: {} for cfg in PROTOCOLS}
execution_order = [(cfg, model_name) for cfg in PROTOCOLS for model_name in MODELS]

for cfg, model_name in execution_order:
    ckpt = load_checkpoint(cfg['slug'], model_name)
    if ckpt is not None:
        print(f"[CHECKPOINT] Skipping {model_name} | {cfg['name']} (already done)")
        result, y_true, y_pred, y_score = ckpt
    else:
        print(f"[RUNNING] {model_name} | {cfg['name']} ...")
        result, y_true, y_pred, y_score, fitted_model = run_experiment(model_name, cfg)
        save_checkpoint(cfg['slug'], model_name, result, y_true, y_pred, y_score, fitted_model)
    print(f"  Accuracy: {result['Accuracy (%)']:.2f}% | Precision: {result['Precision (%)']:.2f}% | "
          f"Recall: {result['Recall (%)']:.2f}% | F1: {result['F1-Score (%)']:.2f}% | "
          f"ROC-AUC: {result['ROC-AUC (%)']:.2f}%")
    all_results.append(result)
    predictions[cfg['slug']][model_name] = (y_true, y_pred, y_score)

protocol_order = {cfg['name']: i for i, cfg in enumerate(PROTOCOLS)}
model_order = {name: i for i, name in enumerate(MODELS)}
results_df = pd.DataFrame(all_results)
results_df['_protocol_order'] = results_df['Experiment'].map(protocol_order)
results_df['_model_order'] = results_df['Model'].map(model_order)
results_df = results_df.sort_values(['_protocol_order', '_model_order']).drop(
    columns=['_protocol_order', '_model_order']).reset_index(drop=True)
# Preserve the original headline column order; traceability columns follow it.
original_columns = ['Model', 'Experiment', 'Split', 'CV', 'Accuracy (%)', 'Precision (%)',
                    'Recall (%)', 'F1-Score (%)', 'ROC-AUC (%)', 'Best_Params', 'CV_F1_Mean (%)']
results_df = results_df[original_columns + [c for c in results_df if c not in original_columns]]
expected_count = len(PROTOCOLS) * len(MODELS)
assert len(results_df) == expected_count
print(f'\n=== ALL {expected_count} EXPERIMENTS COMPLETE ===')
display(results_df[original_columns])


## B4. Full Metrics Table and Classification Reports
*The same report layout and target labels as the original.*


In [ ]:
for cfg in PROTOCOLS:
    for model_name in MODELS:
        y_true, y_pred, y_score = predictions[cfg['slug']][model_name]
        print(f"\n=== Classification Report: {model_name} | {cfg['name']} ===")
        print(classification_report(y_true, y_pred, labels=[0, 1],
              target_names=['Not Bankrupt', 'Bankrupt'], zero_division=0))


## B5. Charts — Four-Model Bars, Confusion Matrices and ROC Curves
Keep the original three-chart-per-protocol output style and filenames: **33 figures per SMOTE ratio**.
Bar charts and ROC curves show all four models. Confusion matrices use a 2×2 panel arrangement;
RF keeps its blue matrix and XGBoost its purple matrix. `Healthy (0)` retains the historical display
label for Not Bankrupt; it is not an independently verified financial-health diagnosis.
CV-only titles identify ordinary held-out CV. SVM ROC curves use decision margins, not probabilities.


In [ ]:
MODEL_COLORS = {'RandomForest': '#4C93E0', 'XGBoost': '#D866F5',
                'LogisticRegression': '#3AA675', 'SVM_RBF': '#E89C36'}
MODEL_CMAPS = {'RandomForest': 'Blues', 'XGBoost': 'Purples',
               'LogisticRegression': 'Greens', 'SVM_RBF': 'Oranges'}
metrics_labels = ['Accuracy (%)', 'ROC-AUC (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)']
for cfg in PROTOCOLS:
    slug, pname = cfg['slug'], cfg['name']
    eval_suffix = ' [Ordinary CV held-out]' if cfg['split'] is None else ''
    rows = {name: results_df[(results_df['Experiment'] == pname) &
                            (results_df['Model'] == name)].iloc[0] for name in MODELS}

    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(metrics_labels))
    width = 0.8 / len(MODELS)
    for model_index, model_name in enumerate(MODELS):
        values = [rows[model_name][metric] for metric in metrics_labels]
        offset = (model_index - (len(MODELS) - 1) / 2) * width
        bars = ax.bar(x + offset, values, width, label=MODEL_LABELS[model_name],
                      color=MODEL_COLORS[model_name])
        for bar in bars:
            ax.annotate(f'{bar.get_height():.1f}%', (bar.get_x()+bar.get_width()/2, bar.get_height()),
                        textcoords='offset points', xytext=(0,3), ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(metrics_labels); ax.set_ylabel('Score (%)'); ax.set_ylim(0,110)
    ax.set_title(f'Model Performance Comparison — {pname}{eval_suffix} (Target: Bankrupt SMEs)', weight='bold')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_bar_comparison.png', dpi=300); plt.show(); plt.close()

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    for ax, model_name in zip(axes.flat, MODELS):
        y_true, y_pred, _ = predictions[slug][model_name]
        cm = confusion_matrix(y_true, y_pred, labels=[0,1])
        ax.imshow(cm, cmap=MODEL_CMAPS[model_name])
        ax.set_title(f'{MODEL_LABELS[model_name]} Confusion Matrix', weight='bold')
        ax.set_xticks([0,1]); ax.set_xticklabels(['Healthy (0)', 'Bankrupt (1)'])
        ax.set_yticks([0,1]); ax.set_yticklabels(['Healthy (0)', 'Bankrupt (1)'])
        ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                        color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
    plt.suptitle(f'Confusion Matrices — {pname}{eval_suffix}', weight='bold'); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_confusion_matrices.png', dpi=300); plt.show(); plt.close()

    fig, ax = plt.subplots(figsize=(8, 6))
    for model_name in MODELS:
        y_true, _, y_score = predictions[slug][model_name]
        fpr, tpr, _ = roc_curve(y_true, y_score)
        ax.plot(fpr, tpr, color=MODEL_COLORS[model_name],
                label=f"{MODEL_LABELS[model_name]} (AUC = {rows[model_name]['ROC-AUC (%)']/100:.4f})")
    ax.plot([0,1], [0,1], linestyle='--', color='gray')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Receiver Operating Characteristic (ROC) Curve — {pname}{eval_suffix}', weight='bold')
    ax.legend(loc='lower right'); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_roc_curve.png', dpi=300); plt.show(); plt.close()
    print(f'Saved 3 charts for: {pname}')


## B6. Save Final Result Tables
The headline slide table keeps the original column layout. Full results retain traceability and fold details.


In [ ]:
results_df.to_csv(f'{report_dir}/master_results_full.csv', index=False)
slide_columns = ['Model', 'Experiment', 'Split', 'CV', 'Accuracy (%)', 'Precision (%)',
                 'Recall (%)', 'F1-Score (%)', 'ROC-AUC (%)', 'CV_F1_Mean (%)']
slide_df = results_df[slide_columns]
slide_df.to_csv(f'{report_dir}/master_results_slide.csv', index=False)
with open(f'{report_dir}/master_results_full.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('=== FINAL RESULTS TABLE ===')
display(slide_df)
print(f'\nAll tables, charts ({3 * len(PROTOCOLS)} images), and model checkpoints ({len(PROTOCOLS) * len(MODELS)} .joblib files) saved to:')
print(f'  {report_dir}')
print('\nReview CV and held-out results before selecting a final model. The previous winner is a candidate, not an assumed new winner.')


## B7. Export a Candidate and Its Fixed 20-Feature Input Schema
The default remains RF 80:20 + CV=10 for compatibility, not an automatic winner.
Change `EXPORT_MODEL_NAME`/`EXPORT_PROTOCOL_SLUG` below to export another evaluated candidate.
For LR/SVM, the 20-feature deployment bundle includes the fitted scaler; it accepts the unscaled
selected feature values in the listed order. The bare classifier is also saved but expects scaled
values for those models. SVM exports class predictions and decision scores, not probabilities.
Exporting does not change the deployed prototype. Each candidate has its own fixed feature manifest.


In [ ]:
EXPORT_PROTOCOL_SLUG = '8020_cv10'
EXPORT_MODEL_NAME = 'RandomForest'
assert EXPORT_MODEL_NAME in MODELS
assert EXPORT_PROTOCOL_SLUG in {cfg['slug'] for cfg in PROTOCOLS}
source_paths = checkpoint_paths(EXPORT_PROTOCOL_SLUG, EXPORT_MODEL_NAME)
chosen_checkpoint = load_checkpoint(EXPORT_PROTOCOL_SLUG, EXPORT_MODEL_NAME)
if chosen_checkpoint is None:
    raise RuntimeError('Run the selected experiment before exporting it.')
chosen_result = chosen_checkpoint[0]
chosen_pipeline = joblib.load(source_paths['model'])
final_selector = chosen_pipeline.named_steps['ig']
final_classifier = chosen_pipeline.named_steps['model']
final_features = selected_features(chosen_pipeline)
assert len(final_features) == K_FEATURES
assert final_classifier.feature_names_in_.tolist() == final_features, 'Classifier input order differs from selector output.'

# Bundle the fitted scaler with the classifier for raw 20-feature inputs.
deployment_steps = []
if 'scale' in chosen_pipeline.named_steps:
    deployment_steps.append(('scale', chosen_pipeline.named_steps['scale']))
deployment_steps.append(('model', final_classifier))
deployment_bundle = SkPipeline(deployment_steps)

# Verify both class predictions and continuous ROC scores against the full pipeline.
verification_input = X_full.iloc[:min(32, len(X_full))]
selected_input = final_selector.transform(verification_input)
full_pred, full_score = predict_outputs(chosen_pipeline, verification_input)
bundle_pred, bundle_score = predict_outputs(deployment_bundle, selected_input)
np.testing.assert_array_equal(full_pred, bundle_pred)
np.testing.assert_allclose(full_score, bundle_score, rtol=1e-10, atol=1e-12)

stem = f'SHIELD_{EXPORT_MODEL_NAME}_{EXPORT_PROTOCOL_SLUG}_SMOTE_{RATIO_TAG}'
full_pipeline_path = f'{model_dir}/{stem}_full_pipeline.joblib'
bundle_path = f'{model_dir}/{stem}_20_feature_deployment.joblib'
classifier_path = f'{model_dir}/{stem}_20_feature_classifier.joblib'
selector_path = f'{model_dir}/{stem}_ig_selector.joblib'
joblib.dump(chosen_pipeline, full_pipeline_path)
joblib.dump(deployment_bundle, bundle_path)
joblib.dump(final_classifier, classifier_path)
joblib.dump(final_selector, selector_path)
manifest = {
    'Run_ID': RUN_ID, 'Model': EXPORT_MODEL_NAME, 'Protocol': EXPORT_PROTOCOL_SLUG,
    'SMOTE_Ratio': float(SMOTE_RATIO), 'Target': TARGET_COL, 'Label_Map': LABEL_MAP,
    'Decision_Rule': 'default classifier.predict(); no threshold tuning',
    'Score_Type': SCORE_TYPES[EXPORT_MODEL_NAME],
    'Score_Is_Probability': EXPORT_MODEL_NAME != 'SVM_RBF',
    'Full_Pipeline_Input_Columns': final_selector.feature_names_in_.tolist(),
    'Deployment_Classifier_Input_Columns': final_features,
    'Deployment_Bundle_Input_Values': 'Unscaled selected features; fitted scaler included when needed',
    'Bare_Classifier_Input_Values': 'Scaled features' if 'scale' in chosen_pipeline.named_steps else 'Unscaled features',
    'Deployment_Steps': [name for name, _ in deployment_steps],
    'Selected_Feature_IG_Scores': {
        str(name): float(score) for name, score, keep in zip(
            final_selector.feature_names_in_, final_selector.scores_, final_selector.get_support()) if keep
    },
    'Experiment_Result': chosen_result,
}
with open(f'{model_dir}/{stem}_feature_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
pd.DataFrame({'Input_Order': range(1, K_FEATURES+1), 'Feature': final_features}).to_csv(
    f'{model_dir}/{stem}_selected_features.csv', index=False)
print(f'SUCCESS: candidate exported ({EXPORT_MODEL_NAME} | {EXPORT_PROTOCOL_SLUG}, SMOTE={SMOTE_RATIO}).')
print('Full pipeline:', full_pipeline_path)
print('20-input deployment bundle:', bundle_path)
print('Bare classifier:', classifier_path)
print('ROC score type:', SCORE_TYPES[EXPORT_MODEL_NAME])
print('Fixed frontend/backend input schema:')
for index, feature in enumerate(final_features, start=1):
    print(f'  {index:2d}. {feature}')
print('This candidate has not replaced the deployed prototype model.')


## Method Notes and References
- Restarted phase: 11 protocols × 4 models = 44 results per ratio; run ratios 0.1 → 0.25 → 0.5 → 1.0 manually.
  Previous two-model/nested-CV results remain a separate phase and are not silently replaced.
- Split-only and ordinary CV-only use predefined fixed parameters. Split+CV uses model-specific grids.
  Their differences include parameter selection as well as partitioning; they do not isolate partition effects.
- CV-only scores come from pooled held-out predictions, not predictions from the final full-data refit.
  Mean fold F1 can differ from pooled F1; fold mean ROC-AUC and individual fold records are also saved.
  Best_Params is a compatibility column; Parameter_Source distinguishes fixed settings from grid winners.
- All IG fitting, scaler fitting and SMOTE resampling use only the applicable training data.
  StandardScaler is added only for LR/SVM, before SMOTE. RF/XGBoost retain their previous unscaled path.
  Scaling also changes SMOTE's neighbour geometry, so comparisons include model-appropriate preprocessing.
- SVM uses default class predictions and decision_function for ROC-AUC. It does not perform probability
  calibration; a decision margin must not be interpreted as a bankruptcy probability or a severity percentage.
- The selected candidate has a fixed 20-feature schema. LR/SVM exports must include their fitted scaler.
  Different fitted candidates may select different feature names; the prototype uses one chosen manifest.
- Seeded splits, versions and configuration/data fingerprints support reproducibility. Logistic Regression
  with lbfgs and SVM without probability calibration do not use random_state for their fitting, but their
  preprocessing/splits are seeded. Change NOTEBOOK_VERSION when manually changing algorithm logic.
- Threshold tuning, prototype severity thresholds and broader RF parameter tuning remain unchanged/on hold.
- CV-only settings must be predefined. Selecting settings/model/SMOTE strength using these evaluation
  scores makes subsequent reported results exploratory. Repeatedly inspected test sets are also exploratory;
  use independent data for confirmatory claims. No split ratio is guaranteed to improve minority-class metrics.
- [Scikit-learn: preprocessing leakage and pipelines](https://scikit-learn.org/stable/common_pitfalls.html)
- [Scikit-learn: cross-validation and holdout evaluation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- [Scikit-learn: SVM scores, scaling and runtime](https://scikit-learn.org/stable/modules/svm.html)
- [Imbalanced-learn: SMOTE sampling ratios](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html)
